In [1]:
# ====== Nettoyage optimisé en lot pour 20 fichiers Parquet (traitement des NA) ======
# - Lit tous les *.parquet dans PROJECT_ROOT
# - Impute automatiquement les NA selon règles métier + statistiques (mode/médiane)
# - Écrit les fichiers nettoyés dans PROJECT_ROOT/cleaned_data
# - Robuste aux colonnes manquantes et aux chemins relatifs

from pathlib import Path
import pandas as pd
import numpy as np
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

# -------- Paramètres --------
PROJECT_ROOT = Path(r"C:\Users\nzigu\Projet Etape 3\Taxi-Copy1")  # dossier contenant vos 20 fichiers .parquet
RAW_GLOB = "*.parquet"                                            # motif des fichiers à traiter
OUT_DIR = PROJECT_ROOT / "cleaned_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

# -------- Constantes métier --------
AIRPORT_LOCATION_IDS = {132, 138}  # JFK, LGA
NEWARK_LOCATION_ID = 1             # Newark -> RatecodeID = 3
DEFAULT_CONG_SUP_POS = 2.5         # secours si pas de médiane dispo
DEFAULT_CONG_SUP_NEG = -2.5

# -------- Utilitaires statistiques sûrs --------
def _mode_safe(s: pd.Series):
    try:
        m = s.mode(dropna=True)
        return m.iloc[0] if len(m) else None
    except Exception:
        return None

def _median_safe(s: pd.Series):
    try:
        return float(s.dropna().median()) if s.size else None
    except Exception:
        return None

def _fillna_with_mode(df: pd.DataFrame, col: str):
    if col in df.columns:
        v = _mode_safe(df[col])
        if v is not None:
            df[col] = df[col].fillna(v)

# -------- Manhattan IDs depuis taxi_zone_lookup.csv (si dispo) --------
def load_manhattan_ids() -> set:
    try:
        zone = pd.read_csv('taxi_zone_lookup.csv')
        if {'LocationID','Borough'}.issubset(zone.columns):
            return set(
                zone.loc[zone['Borough'].astype(str).str.upper()=='MANHATTAN', 'LocationID']
                    .dropna().astype(int).tolist()
            )
    except FileNotFoundError:
        pass
    return set()

MANHATTAN_IDS = load_manhattan_ids()

# -------- Construction dynamique des lieux où la surcharge s'applique --------
def compute_all_location(df: pd.DataFrame) -> set:
    base = set(MANHATTAN_IDS)
    need = {'PULocationID','DOLocationID','congestion_surcharge'}
    if not need.issubset(df.columns):
        return base
    surcharge_df = df[df['congestion_surcharge'].fillna(0) > 0]
    if surcharge_df.empty:
        return base
    hors_manhattan = surcharge_df[
        ~surcharge_df['PULocationID'].isin(base) &
        ~surcharge_df['DOLocationID'].isin(base)
    ]
    if hors_manhattan.empty:
        return base
    location = pd.unique(pd.concat(
        [hors_manhattan['PULocationID'], hors_manhattan['DOLocationID']],
        ignore_index=True
    ))
    extra = set(map(int, pd.Series(location).dropna().astype(int)))
    return base | extra

# -------- Règles d’imputation des NA --------
def custom_clean(df: pd.DataFrame) -> pd.DataFrame:
    # 1) passenger_count -> mode
    if 'passenger_count' in df.columns:
        _fillna_with_mode(df, 'passenger_count')

    # 2) RatecodeID : aéroports puis médiane
    if 'RatecodeID' in df.columns and {'PULocationID','DOLocationID'}.issubset(df.columns):
        df.loc[
            (df['RatecodeID'].isna()) &
            (df['PULocationID'].isin(AIRPORT_LOCATION_IDS) | df['DOLocationID'].isin(AIRPORT_LOCATION_IDS)),
            'RatecodeID'
        ] = 2
        df.loc[
            (df['RatecodeID'].isna()) &
            ((df['PULocationID'] == NEWARK_LOCATION_ID) | (df['DOLocationID'] == NEWARK_LOCATION_ID)),
            'RatecodeID'
        ] = 3
        med_rate = _median_safe(df['RatecodeID'])
        if med_rate is not None:
            df['RatecodeID'] = df['RatecodeID'].fillna(med_rate)

    # 3) store_and_fwd_flag -> mode
    if 'store_and_fwd_flag' in df.columns:
        _fillna_with_mode(df, 'store_and_fwd_flag')

    # 4) congestion_surcharge : dépend de Manhattan/all_location + signe de total_amount
    if {'congestion_surcharge','PULocationID','DOLocationID'}.issubset(df.columns):
        all_location = compute_all_location(df)

        if len(all_location) > 0:
            mask_out = (~df['PULocationID'].isin(all_location)) & (~df['DOLocationID'].isin(all_location))
            df.loc[mask_out, 'congestion_surcharge'] = df.loc[mask_out, 'congestion_surcharge'].fillna(0)

        if 'total_amount' in df.columns and len(all_location) > 0:
            mask_in = (df['PULocationID'].isin(all_location)) | (df['DOLocationID'].isin(all_location))
            med_sup = _median_safe(df.loc[df['total_amount'] > 0, 'congestion_surcharge']) or DEFAULT_CONG_SUP_POS
            med_inf = _median_safe(df.loc[df['total_amount'] < 0, 'congestion_surcharge']) or DEFAULT_CONG_SUP_NEG
            df.loc[mask_in & df['congestion_surcharge'].isna() & (df['total_amount'] > 0), 'congestion_surcharge'] = med_sup
            df.loc[mask_in & df['congestion_surcharge'].isna() & (df['total_amount'] < 0), 'congestion_surcharge'] = med_inf

    # 5) Airport_fee : 0 hors aéroport, sinon médianes conditionnelles
    if 'Airport_fee' in df.columns and {'PULocationID','DOLocationID'}.issubset(df.columns):
        mask_no_air = (~df['PULocationID'].isin(AIRPORT_LOCATION_IDS)) & (~df['DOLocationID'].isin(AIRPORT_LOCATION_IDS))
        df.loc[mask_no_air, 'Airport_fee'] = df.loc[mask_no_air, 'Airport_fee'].fillna(0)
        if 'total_amount' in df.columns:
            med_sup_airp = _median_safe(df.loc[df['total_amount'] > 0, 'Airport_fee'])
            med_inf_airp = _median_safe(df.loc[df['total_amount'] < 0, 'Airport_fee'])
            if med_sup_airp is None:
                med_sup_airp = _median_safe(df['Airport_fee']) or 0
            if med_inf_airp is None:
                med_inf_airp = 0
            mask_air = (df['PULocationID'].isin(AIRPORT_LOCATION_IDS)) | (df['DOLocationID'].isin(AIRPORT_LOCATION_IDS))
            df.loc[mask_air & df['Airport_fee'].isna() & (df['total_amount'] > 0), 'Airport_fee'] = med_sup_airp
            df.loc[mask_air & df['Airport_fee'].isna() & (df['total_amount'] < 0), 'Airport_fee'] = med_inf_airp

    # 6) Colonnes numériques courantes : remplissage final optionnel (médiane) si NA persistent
    for col in ['fare_amount','extra','mta_tax','tip_amount','tolls_amount','improvement_surcharge','total_amount']:
        if col in df.columns and df[col].isna().any():
            med = _median_safe(df[col])
            if med is not None:
                df[col] = df[col].fillna(med)

    return df

# -------- Traitement d’un fichier --------
def process_file(path: Path) -> Path:
    df = pd.read_parquet(path)
    df_clean = custom_clean(df)
    out_path = OUT_DIR / path.name
    df_clean.to_parquet(out_path, index=False)
    return out_path

# -------- Lancement en lot (threading pour I/O) --------
def main():
    files = sorted(PROJECT_ROOT.glob(RAW_GLOB))
    if not files:
        print("Aucun fichier Parquet trouvé dans", PROJECT_ROOT)
        return
    print(f"{len(files)} fichier(s) à traiter")
    done = 0
    with ThreadPoolExecutor(max_workers=min(8, len(files))) as ex:
        futs = {ex.submit(process_file, p): p.name for p in files}
        for fut in as_completed(futs):
            outp = fut.result()
            done += 1
            print(f"[{done}/{len(files)}] OK -> {outp.name}")
    print("Terminé. Sortie:", OUT_DIR.resolve())

if __name__ == "__main__":
    main()


19 fichier(s) à traiter
[1/19] OK -> Taxi_2024-1.parquet
[2/19] OK -> Taxi_2024-08.parquet
[3/19] OK -> Taxi_2024-07.parquet
[4/19] OK -> Taxi_2024-06.parquet
[5/19] OK -> Taxi_2024-11.parquet
[6/19] OK -> Taxi_2024-09.parquet
[7/19] OK -> Taxi_2024-12.parquet
[8/19] OK -> Taxi_2024-10.parquet
[9/19] OK -> Taxi_2024-2.parquet
[10/19] OK -> Taxi_2024-3.parquet
[11/19] OK -> Taxi_2024-4.parquet
[12/19] OK -> Taxi_2024-5.parquet
[13/19] OK -> Taxi_2025-01.parquet
[14/19] OK -> Taxi_2025-02.parquet
[15/19] OK -> Taxi_2025-04.parquet
[16/19] OK -> Taxi_2025-03.parquet
[17/19] OK -> Taxi_2025-05.parquet
[18/19] OK -> Taxi_2025-06.parquet
[19/19] OK -> Taxi_2025-07.parquet
Terminé. Sortie: C:\Users\nzigu\Projet Etape 3\Taxi-Copy1\cleaned_data
